In [1]:
import sys
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Thiết lập đường dẫn hệ thống
root = os.path.abspath(os.path.join(".."))
if root not in sys.path:
    sys.path.append(root)

from src.data.cleaner import preprocess_for_mining

# 2. Nạp dữ liệu đã làm sạch từ file 02
df_cleaned = pd.read_csv("../data/processed/creditcard_cleaned.csv")

# 3. Lấy mẫu (Sampling) để chạy nhanh và tránh treo máy
fraud_df = df_cleaned[df_cleaned['Class'] == 1]
normal_df = df_cleaned[df_cleaned['Class'] == 0].sample(n=1000, random_state=42)
df_sampling = pd.concat([fraud_df, normal_df])

# 4. Tiền xử lý dữ liệu cho khai phá (Chỉ lấy cột nhãn)
df_mining_ready = preprocess_for_mining(df_sampling)

# 5. One-hot encoding và ép kiểu BOOLEAN (Sửa lỗi ValueError triệt để)
df_encoded = pd.get_dummies(df_mining_ready).astype(bool)

# 6. Chạy thuật toán Apriori
# Tìm các tập phổ biến xuất hiện ít nhất 5% trong tập dữ liệu mẫu
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)

# 7. Trích xuất luật kết hợp
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 8. Lọc các luật dẫn đến kết quả Gian lận (Status_FRAUD)
fraud_rules = rules[rules['consequents'].apply(lambda x: 'Status_FRAUD' in str(x))]
fraud_rules = fraud_rules.sort_values(by='confidence', ascending=False)

# 9. Lưu kết quả vào outputs để Dashboard hiển thị
output_path = "../outputs/tables/fraud_rules.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
fraud_rules.to_csv(output_path, index=False)

print(f"✅ Thành công! Đã tạo {len(fraud_rules)} luật kết hợp.")
display(fraud_rules.head(10))

✅ Thành công! Đã tạo 2 luật kết hợp.


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(Amount_Bin_Low),(Status_Status_FRAUD),0.333333,0.321113,0.146640,0.439919,1.369979,1.0,0.039602,1.212121,0.405093,0.288770,0.175,0.448289
4,(Amount_Bin_High),(Status_Status_FRAUD),0.333333,0.321113,0.121521,0.364562,1.135307,1.0,0.014483,1.068376,0.178771,0.228025,0.064,0.371499
